# 0. Import library

In [1]:
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score


# --- Random search for best LightGBM params (uses your fixed validation set) ---
import time
import copy
from sklearn.model_selection import ParameterSampler

In [2]:
import kagglehub

In [3]:
import kagglehub, kagglesdk
print(kagglehub.__version__)
print(kagglesdk.__version__)

1.0.1
0.1.23


In [4]:
kagglehub.login()

In [15]:
K_PATH = kagglehub.competition_download("sem-eval-2026-task-13-subtask-a")
#K_PATH = '/kaggle/input/competitions/sem-eval-2026-task-13-subtask-a'
print('~'*70)
print('data_dir :', K_PATH)
print('~'*70)

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
data_dir : /home/dxk/.cache/kagglehub/competitions/sem-eval-2026-task-13-subtask-a
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~


# 1. Feature Extraction

In [16]:
FEATURE_NAMES = [
    # Base features
    'avg_line_len', 
    'std_line_len', 
    'avg_indent', 
    'std_indent', 
    'max_indent',
    'empty_gap_std',
    'empty_gap_mean',
    
    # Comment features
    'n_single_comments',
    # 'n_block_comments',
    # 'avg_comment_len',
    # 'comment_ratio',
    # 'comment_density',
    'text_comment_ratio',
    # 'code_comment_ratio',
    
    # Variable convention & length features
    # 'camel_case_ratio',
    # 'snake_case_ratio',
    # 'pascal_case_ratio',
    # 'screaming_snake_ratio',
    # 'short_id_ratio',
    # 'long_id_ratio',
    # 'avg_id_len',
]

print('~'*70)
print('Handcraft Features defined!')
print('~'*70)

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Handcraft Features defined!
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~


In [17]:
tqdm.pandas()

In [18]:
import os
from utils import parallel_extract

TRAIN_PREPROCESS_PATH = 'train_preprocess.parquet'
TEST_PREPROCESS_PATH = 'test_preprocess.parquet'
VAL_PREPROCESS_PATH = 'val_preprocess.parquet'
RAW_TRAIN_PATH = os.path.join(K_PATH, 'Task_A', 'train.parquet')
RAW_TEST_PATH = os.path.join(K_PATH, 'Task_A', 'test.parquet')
RAW_VAL_PATH = os.path.join(K_PATH, 'Task_A', 'validation.parquet')

    
print('Reading train_df and test_df from parquet...')
train_df = pd.read_parquet(RAW_TRAIN_PATH)
test_df = pd.read_parquet(RAW_TEST_PATH)
val_df = pd.read_parquet(RAW_VAL_PATH)




if os.path.exists(TRAIN_PREPROCESS_PATH) and os.path.exists(TEST_PREPROCESS_PATH) and os.path.exists(VAL_PREPROCESS_PATH):
    print(f'Already pre-extracted features from {TRAIN_PREPROCESS_PATH} and {TEST_PREPROCESS_PATH}...')
    
else:
    print(f'Extracting {len(FEATURE_NAMES)} handcrafted features in parallel...')

    # Chạy song song thay vì dùng list comprehension
    code_train_list = train_df['code'].to_list()
    X_hand_train = parallel_extract(code_train_list, FEATURE_NAMES)
    y_train = train_df['label'].to_numpy()
    print(f'Handcrafted features: train={X_hand_train.shape}')
    
    code_test_list = test_df['code'].to_list()
    X_hand_test = parallel_extract(code_test_list, FEATURE_NAMES)
    print(f'Handcrafted features: test={X_hand_test.shape}')
    
    code_val_list = val_df['code'].to_list()
    X_hand_val = parallel_extract(code_val_list, FEATURE_NAMES)
    print(f'Handcrafted features: val={X_hand_val.shape}')
    

    # 1. Chuyển Numpy Array thành DataFrame
    train_features_df = pd.DataFrame(X_hand_train, columns=FEATURE_NAMES)
    test_features_df = pd.DataFrame(X_hand_test, columns=FEATURE_NAMES)
    val_features_df = pd.DataFrame(X_hand_val, columns=FEATURE_NAMES)
    
    # 2. Reset index của DF gốc để đảm bảo nối đúng dòng (rất quan trọng)
    train_df = train_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)
    val_df = val_df.reset_index(drop=True)
    
    # 3. Nối ngang (axis=1) DataFrame gốc với DataFrame đặc trưng
    train_df = pd.concat([train_df, train_features_df], axis=1)
    test_df = pd.concat([test_df, test_features_df], axis=1)
    val_df = pd.concat([val_df, val_features_df], axis=1)
    
    print(f"Kích thước mới của dữ liệu: train={train_df.shape}, test={test_df.shape}, val={val_df.shape}")

    print(f'Saving full combined data to {TRAIN_PREPROCESS_PATH}, {TEST_PREPROCESS_PATH} and {VAL_PREPROCESS_PATH}...')
    train_df.to_parquet(TRAIN_PREPROCESS_PATH, index=False)
    test_df.to_parquet(TEST_PREPROCESS_PATH, index=False)
    val_df.to_parquet(VAL_PREPROCESS_PATH, index=False)
    


Reading train_df and test_df from parquet...
Already pre-extracted features from train_preprocess.parquet and test_preprocess.parquet...


# 2. Create helper functions

In [32]:
import subprocess
def submit_solution(filename):
    # filename = 'submission.csv'
    # Định nghĩa lệnh cần chạy dưới dạng một list các chuỗi
    command = [
        "kaggle", 
        "competitions", 
        "submit", 
        "-c", "sem-eval-2026-task-13-subtask-a", 
        "-f", f"{filename}", 
        "-m", filename.replace('.csv', '')
    ]

    try:
        # Chạy lệnh
        result = subprocess.run(command, capture_output=True, text=True, check=True)
        
        # In ra kết quả thành công từ Kaggle
        print("Nộp bài thành công!")
        print(result.stdout)
        
    except subprocess.CalledProcessError as e:
        # Bắt lỗi nếu lệnh CLI thất bại (ví dụ: sai tên file, chưa authenticate)
            print("Có lỗi xảy ra khi nộp bài:")
            print(e.stderr)

In [33]:
def create_file_submission(test_preds, threshold, filename):

    submit_preds = (test_preds > threshold).astype(int)

    submission = pd.DataFrame({
        'ID': test_pre_df['ID'],
        'label': submit_preds
    })
    submission.to_csv(filename, index=False)

    print("Distribution of predictions in submission:")
    print(submission['label'].value_counts(normalize=True) * 100)
    
    submit_solution(filename)


## 2.1 LightGBM model 

In [21]:
def lgb_macro_f1(y_true, y_pred):
    y_pred_bin = (y_pred > 0.5).astype(int)
    f1 = f1_score(y_true, y_pred_bin, average='macro')
    return 'macro_f1', f1, True

In [22]:


def train_lightgbm(train, val, test, features, lgb_params):
    """
    Huấn luyện 1 mô hình LightGBM duy nhất trên toàn bộ tập train.
    Dùng tập val để đánh giá Early Stopping.
    """
    print("Bắt đầu huấn luyện mô hình duy nhất (No K-Fold)...")
    
    # 1. Trích xuất features
    X_train = train[features]
    y_train = train['label']
    
    X_val = val[features]  
    y_val = val['label']
    
    X_test = test[features] 
    
    # Khởi tạo mảng lưu kết quả dự đoán
    # Thay oof_preds thành train_preds để bạn có thể kiểm tra xem mô hình có bị overfit trên chính nó không
    train_preds = np.zeros(len(train))
    val_preds = np.zeros(len(val))
    test_preds = np.zeros(len(test))
    
    # 2. Khởi tạo và fit mô hình
    model = lgb.LGBMClassifier(**lgb_params)
    
    print("\nĐang fit mô hình... (Theo dõi Macro F1 trên tập Validation)")
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        eval_metric=lgb_macro_f1, # Hàm custom của bạn
        # Set verbose=True hoặc 10 để xem tiến trình học và điểm số thay đổi ra sao
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=True)] 
    )
    
    # 3. Dự đoán xác suất lớp Positive (class 1)
    print("\nĐang tiến hành dự đoán...")
    train_preds = model.predict_proba(X_train)[:, 1]
    val_preds = model.predict_proba(X_val)[:, 1]
    test_preds = model.predict_proba(X_test)[:, 1]
        
    print("\nQuá trình huấn luyện và dự đoán hoàn tất!")
    
    # Trả về model trực tiếp thay vì list
    return model, train_preds, val_preds, test_preds

# ==========================================
# CÁCH SỬ DỤNG
# ==========================================

# Parameters specifically chosen to prevent OOD memorization (Giữ nguyên)
lgb_params = {
    'objective': 'binary',
    'learning_rate': 0.05,
    'max_depth': 5,
    'num_leaves': 31,
    'colsample_bytree': 0.7, 
    'subsample': 0.8,
    'random_state': 42,
    'n_estimators': 500,
    'verbose': -1
}



## 2.2 CatBoost

In [23]:
# 2.2 CatBoost
import importlib
import sys
import subprocess
from sklearn.metrics import f1_score

def _import_or_install(module_name, pip_name=None):
    """Import a module; if missing, pip-install it then import again."""
    try:
        return importlib.import_module(module_name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", (pip_name or module_name)])
        return importlib.import_module(module_name)

catboost = _import_or_install("catboost")
CatBoostClassifier = catboost.CatBoostClassifier

def train_catboost(train, val, test, features, cb_params, threshold=0.9):
    """
    Train CatBoost on train, early-stop on val, predict probabilities for train/val/test.
    Returns: model, train_probs, val_probs, test_probs
    """
    X_train = train[features]
    y_train = train['label']
    X_val = val[features]
    y_val = val['label']
    X_test = test[features]

    model = CatBoostClassifier(**cb_params)
    model.fit(
        X_train, y_train,
        eval_set=(X_val, y_val),
        use_best_model=True,
    )

    train_probs = model.predict_proba(X_train)[:, 1]
    val_probs = model.predict_proba(X_val)[:, 1]
    test_probs = model.predict_proba(X_test)[:, 1]

    val_pred = (val_probs > threshold).astype(int)
    print(f"CatBoost | Val Macro F1 @ thr={threshold:.2f}: {f1_score(y_val, val_pred, average='macro'):.5f}")
    return model, train_probs, val_probs, test_probs

# Default CatBoost params (start point; tune later if needed)
cb_params = {
    'loss_function': 'Logloss',
    'eval_metric': 'F1',  # internal early-stopping metric; we'll still report macro-F1 ourselves
    'iterations': 5000,
    'learning_rate': 0.05,
    'depth': 6,
    'l2_leaf_reg': 3.0,
    'random_seed': 42,
    'od_type': 'Iter',
    'od_wait': 100,
    'thread_count': -1,
    'allow_writing_files': False,
    'verbose': 200,
}

## 2.3 XGBoost

In [24]:
# 2.3 XGBoost
import importlib
import sys
import subprocess
from sklearn.metrics import f1_score

def _import_or_install(module_name, pip_name=None):
    """Import a module; if missing, pip-install it then import again."""
    try:
        return importlib.import_module(module_name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", (pip_name or module_name)])
        return importlib.import_module(module_name)

xgboost = _import_or_install("xgboost")
XGBClassifier = xgboost.XGBClassifier

def train_xgboost(train, val, test, features, xgb_params, threshold=0.9, early_stopping_rounds=100):
    """
    Train XGBoost on train, early-stop on val, predict probabilities for train/val/test.
    Returns: model, train_probs, val_probs, test_probs
    """
    X_train = train[features]
    y_train = train['label']
    X_val = val[features]
    y_val = val['label']
    X_test = test[features]



    model = XGBClassifier(
    **xgb_params,
    eval_metric='logloss',
    early_stopping_rounds=early_stopping_rounds,
)

    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False,
    )


    # model = XGBClassifier(**xgb_params)
    # model.fit(
    #     X_train, y_train,
    #     eval_set=[(X_val, y_val)],
    #     eval_metric='logloss',
    #     verbose=False,
    #     early_stopping_rounds=early_stopping_rounds,
    # )

    train_probs = model.predict_proba(X_train)[:, 1]
    val_probs = model.predict_proba(X_val)[:, 1]
    test_probs = model.predict_proba(X_test)[:, 1]

    val_pred = (val_probs > threshold).astype(int)
    print(f"XGBoost | Val Macro F1 @ thr={threshold:.2f}: {f1_score(y_val, val_pred, average='macro'):.5f}")
    return model, train_probs, val_probs, test_probs

# Default XGBoost params (start point; tune later if needed)
xgb_params = {
    'objective': 'binary:logistic',
    'learning_rate': 0.05,
    'max_depth': 5,
    'n_estimators': 5000,  # early stopping will pick the effective #trees
    'subsample': 0.8,
    'colsample_bytree': 0.7,
    'min_child_weight': 1.0,
    'gamma': 0.0,
    'reg_alpha': 0.0,
    'reg_lambda': 1.0,
    'random_state': 42,
    'n_jobs': -1,
    'tree_method': 'hist',
}

## 2.4 ExtraTrees (sklearn)

In [25]:
# 2.4 ExtraTrees (sklearn)
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import f1_score

def train_extratrees(train, val, test, features, et_params, threshold=0.9):
    """
    Train ExtraTrees on train, evaluate on val, predict probabilities for train/val/test.
    Returns: model, train_probs, val_probs, test_probs
    """
    X_train = train[features]
    y_train = train['label']
    X_val = val[features]
    y_val = val['label']
    X_test = test[features]

    model = ExtraTreesClassifier(**et_params)
    model.fit(X_train, y_train)

    train_probs = model.predict_proba(X_train)[:, 1]
    val_probs = model.predict_proba(X_val)[:, 1]
    test_probs = model.predict_proba(X_test)[:, 1]

    val_pred = (val_probs > threshold).astype(int)
    print(f"ExtraTrees | Val Macro F1 @ thr={threshold:.2f}: {f1_score(y_val, val_pred, average='macro'):.5f}")
    return model, train_probs, val_probs, test_probs

# Default ExtraTrees params (good strong tabular baseline)
et_params = {
    'n_estimators': 5000,
    'random_state': 42,
    'n_jobs': -1,
    'max_depth': 12,
    'max_features': 'sqrt',
    'min_samples_split': 2,
    'min_samples_leaf': 5,
    'bootstrap': False,
    'class_weight': 'balanced_subsample',
}

## 2.5 Logistic Regression (baseline)

In [26]:
# 2.5 Logistic Regression (baseline)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

def train_logreg(train, val, test, features, lr_params, threshold=0.9):
    """
    Train Logistic Regression (with scaling) and output probabilities for train/val/test.
    Returns: model, train_probs, val_probs, test_probs
    """
    X_train = train[features]
    y_train = train['label']
    X_val = val[features]
    y_val = val['label']
    X_test = test[features]

    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(**lr_params),
    )
    model.fit(X_train, y_train)

    train_probs = model.predict_proba(X_train)[:, 1]
    val_probs = model.predict_proba(X_val)[:, 1]
    test_probs = model.predict_proba(X_test)[:, 1]

    val_pred = (val_probs > threshold).astype(int)
    print(f"LogReg | Val Macro F1 @ thr={threshold:.2f}: {f1_score(y_val, val_pred, average='macro'):.5f}")
    return model, train_probs, val_probs, test_probs

# Default Logistic Regression params
lr_params = {
    'C': 1.0,
    'solver': 'liblinear',
    'class_weight': 'balanced',
    'max_iter': 1000,
    'random_state': 42,
}

# 3. Model training

In [27]:
train_pre_df = pd.read_parquet(TRAIN_PREPROCESS_PATH)
test_pre_df = pd.read_parquet(TEST_PREPROCESS_PATH)
val_pre_df = pd.read_parquet(VAL_PREPROCESS_PATH)

## 3.1 LightGBM

In [ ]:
models, oof_preds, val_preds, test_preds = train_lightgbm(train_pre_df, val_pre_df, test_pre_df, FEATURE_NAMES, lgb_params)

Index(['code', 'generator', 'label', 'language', 'avg_line_len',
       'std_line_len', 'avg_indent', 'std_indent', 'max_indent',
       'empty_gap_std', 'empty_gap_mean', 'n_single_comments',
       'n_block_comments', 'avg_comment_len', 'comment_ratio',
       'comment_density', 'text_comment_ratio', 'code_comment_ratio',
       'camel_case_ratio', 'snake_case_ratio', 'pascal_case_ratio',
       'screaming_snake_ratio', 'short_id_ratio', 'long_id_ratio',
       'avg_id_len'],
      dtype='object')
Bắt đầu huấn luyện mô hình duy nhất (No K-Fold)...

Đang fit mô hình... (Theo dõi Macro F1 trên tập Validation)
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[500]	valid_0's binary_logloss: 0.154379	valid_0's macro_f1: 0.943796

Đang tiến hành dự đoán...

Quá trình huấn luyện và dự đoán hoàn tất!


## 3.2 CatBoost

In [65]:
cat_model, cat_train_preds, cat_val_preds, cat_test_preds = train_catboost(train_pre_df, val_pre_df, test_pre_df, FEATURE_NAMES, cb_params)


0:	learn: 0.9104160	test: 0.9104382	best: 0.9104382 (0)	total: 96ms	remaining: 8m
200:	learn: 0.9417539	test: 0.9412135	best: 0.9412135 (200)	total: 8.84s	remaining: 3m 31s
400:	learn: 0.9456184	test: 0.9446862	best: 0.9446862 (400)	total: 17.7s	remaining: 3m 22s
600:	learn: 0.9475544	test: 0.9461620	best: 0.9461620 (600)	total: 26.6s	remaining: 3m 15s
800:	learn: 0.9486307	test: 0.9470707	best: 0.9471090 (799)	total: 37.3s	remaining: 3m 15s
1000:	learn: 0.9495514	test: 0.9474398	best: 0.9475565 (969)	total: 50.2s	remaining: 3m 20s
1200:	learn: 0.9502410	test: 0.9480162	best: 0.9480444 (1148)	total: 1m 3s	remaining: 3m 20s
1400:	learn: 0.9507639	test: 0.9482396	best: 0.9482658 (1397)	total: 1m 15s	remaining: 3m 13s
1600:	learn: 0.9512032	test: 0.9485364	best: 0.9486550 (1571)	total: 1m 26s	remaining: 3m 3s
1800:	learn: 0.9517271	test: 0.9490106	best: 0.9490408 (1764)	total: 1m 37s	remaining: 2m 53s
2000:	learn: 0.9521541	test: 0.9491616	best: 0.9491999 (1993)	total: 1m 50s	remaining: 2

In [70]:
create_file_submission(cat_test_preds, 0.98, "cat98.csv")


Distribution of predictions in submission:
label
0    75.9128
1    24.0872
Name: proportion, dtype: float64
Nộp bài thành công!
Successfully submitted to SemEval-2026-Task13-Subtask-A


## 3.3 XGBoost

In [48]:
import xgboost as xgb
print(xgb.__version__)

3.2.0


In [50]:

xgb_model, train_xgb_preds, val_xgb_preds, test_xgb_preds = train_xgboost(train_pre_df, val_pre_df, test_pre_df, FEATURE_NAMES, xgb_params)

XGBoost | Val Macro F1 @ thr=0.90: 0.91382


In [63]:
create_file_submission(test_xgb_preds, 0.978, "xgb978.csv")

Distribution of predictions in submission:
label
0    71.9938
1    28.0062
Name: proportion, dtype: float64
Nộp bài thành công!
Successfully submitted to SemEval-2026-Task13-Subtask-A


## 3.4 ExtraTrees

In [28]:
# Train + validate ExtraTrees (similar flow to LightGBM/CatBoost/XGBoost)
import numpy as np
from sklearn.metrics import f1_score

if 'train_extratrees' not in globals():
    raise NameError("train_extratrees not found. Run the helper section 2.4 ExtraTrees first.")

def find_best_threshold(y_true, probs, thresholds=None):
    if thresholds is None:
        thresholds = np.linspace(0.80, 0.995, 30)
    best_thr, best_f1 = 0.5, -1.0
    y_true = np.asarray(y_true)
    for thr in thresholds:
        pred = (probs > thr).astype(int)
        score = f1_score(y_true, pred, average='macro')
        if score > best_f1:
            best_f1 = score
            best_thr = float(thr)
    return best_thr, float(best_f1)

def write_submission_file(test_probs, threshold, filename):
    submit_preds = (test_probs > threshold).astype(int)
    submission = pd.DataFrame({
        'ID': test_pre_df['ID'],
        'label': submit_preds,
    })
    submission.to_csv(filename, index=False)
    print("Saved:", filename)
    print("Distribution (%):")
    print(submission['label'].value_counts(normalize=True) * 100)
    return submission

et_model, et_train_probs, et_val_probs, et_test_probs = train_extratrees(
    train_pre_df, val_pre_df, test_pre_df, FEATURE_NAMES, et_params, threshold=0.9
 )

best_thr_et, best_f1_et = find_best_threshold(val_pre_df['label'], et_val_probs)
print(f"ExtraTrees | Best Val Macro F1={best_f1_et:.5f} at thr={best_thr_et:.3f}")

# Write CSV (does NOT auto-submit). If you want submit: submit_solution('extratrees_best.csv')
write_submission_file(et_test_probs, best_thr_et, "extratrees_best.csv")

ExtraTrees | Val Macro F1 @ thr=0.90: 0.34078
ExtraTrees | Best Val Macro F1=0.63902 at thr=0.800
Saved: extratrees_best.csv
Distribution (%):
label
0    77.8432
1    22.1568
Name: proportion, dtype: float64


,ID,label
0,0,0
1,2,0
2,5,1
3,6,0
4,7,0
...,...,...
499995,1108199,0
499996,1108200,0
499997,1108201,0
499998,1108203,1


In [37]:
create_file_submission(et_test_probs, 0.83, "XTrees83.csv")

Distribution of predictions in submission:
label
0    82.4648
1    17.5352
Name: proportion, dtype: float64
Có lỗi xảy ra khi nộp bài:



## 3.5 Logistic Regression

In [39]:
# Train + validate Logistic Regression (baseline)
import numpy as np
from sklearn.metrics import f1_score

if 'train_logreg' not in globals():
    raise NameError("train_logreg not found. Run the helper section 2.5 Logistic Regression first.")

lr_model, lr_train_probs, lr_val_probs, lr_test_probs = train_logreg(
    train_pre_df, val_pre_df, test_pre_df, FEATURE_NAMES, lr_params, threshold=0.8
 )

best_thr_lr, best_f1_lr = find_best_threshold(val_pre_df['label'], lr_val_probs)
print(f"LogReg | Best Val Macro F1={best_f1_lr:.5f} at thr={best_thr_lr:.3f}")

# Write CSV (does NOT auto-submit). If you want submit: submit_solution('logreg_best.csv')
write_submission_file(lr_test_probs, best_thr_lr, "logreg_best.csv")

LogReg | Val Macro F1 @ thr=0.80: 0.79183
LogReg | Best Val Macro F1=0.79183 at thr=0.800
Saved: logreg_best.csv
Distribution (%):
label
0    66.305
1    33.695
Name: proportion, dtype: float64


,ID,label
0,0,0
1,2,0
2,5,1
3,6,0
4,7,0
...,...,...
499995,1108199,1
499996,1108200,0
499997,1108201,0
499998,1108203,1


In [43]:
create_file_submission(lr_test_probs, 0.82, "LogisticRegression82.csv")


Distribution of predictions in submission:
label
0    68.2108
1    31.7892
Name: proportion, dtype: float64
Có lỗi xảy ra khi nộp bài:



In [ ]:


# # Fallback: define metric if you didn't run the previous cell
# if 'lgb_macro_f1' not in globals():
#     def lgb_macro_f1(y_true, y_pred):
#         y_pred_bin = (y_pred > 0.5).astype(int)
#         f1 = f1_score(y_true, y_pred_bin, average='macro')
#         return 'macro_f1', f1, True

# # Safety: load dataframes if the previous cells weren't run
# if 'train_pre_df' not in globals():
#     train_pre_df = pd.read_parquet(TRAIN_PREPROCESS_PATH)
# if 'val_pre_df' not in globals():
#     val_pre_df = pd.read_parquet(VAL_PREPROCESS_PATH)
# if 'test_pre_df' not in globals():
#     test_pre_df = pd.read_parquet(TEST_PREPROCESS_PATH)

# X_train = train_pre_df[FEATURE_NAMES]
# y_train = train_pre_df['label']
# X_val = val_pre_df[FEATURE_NAMES]
# y_val = val_pre_df['label']

In [44]:


# BASE_PARAMS = {
#     'objective': 'binary',
#     'random_state': 42,
#     'n_estimators': 5000,  # let early stopping choose the effective #trees
#     'n_jobs': -1,
#     'verbose': -1,
# }

# # Discrete distributions (no SciPy needed)
# param_distributions = {
#     'learning_rate': [0.005, 0.01, 0.02, 0.03, 0.05, 0.07],
#     'max_depth': [-1, 3, 4, 5, 6, 8],
#     'num_leaves': [15, 31, 63, 127],
#     'min_child_samples': [5, 10, 20, 30, 50, 100],
#     'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
#     'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
#     'reg_alpha': [0.0, 1e-3, 1e-2, 1e-1, 1.0],
#     'reg_lambda': [0.0, 1e-3, 1e-2, 1e-1, 1.0],
# }

# N_ITER = 10
# EARLY_STOPPING_ROUNDS = 100
# SEED = 42

# sampler = ParameterSampler(param_distributions, n_iter=N_ITER, random_state=SEED)

# best = {
#     'macro_f1': -1.0,
#     'params': None,
#     'best_iteration': None,
# }
# results = []

# t0 = time.time()
# print(f"Running random search: N_ITER={N_ITER}, early_stopping={EARLY_STOPPING_ROUNDS}")
# for i, sampled in enumerate(sampler, start=1):
#     params = copy.deepcopy(BASE_PARAMS)
#     params.update(sampled)

#     model_rs = lgb.LGBMClassifier(**params)
#     model_rs.fit(
#         X_train, y_train,
#         eval_set=[(X_val, y_val)],
#         eval_metric=lgb_macro_f1,
#         callbacks=[lgb.early_stopping(stopping_rounds=EARLY_STOPPING_ROUNDS, verbose=False)],
#     )

#     val_probs = model_rs.predict_proba(X_val)[:, 1]
#     val_pred = (val_probs > 0.5).astype(int)
#     macro_f1 = f1_score(y_val, val_pred, average='macro')
#     best_iter = getattr(model_rs, 'best_iteration_', None)

#     results.append({
#         **sampled,
#         'macro_f1': float(macro_f1),
#         'best_iteration': int(best_iter) if best_iter is not None else None,
#     })

#     if macro_f1 > best['macro_f1']:
#         best.update({
#             'macro_f1': float(macro_f1),
#             'params': params,
#             'best_iteration': best_iter,
#         })

#     if i % 5 == 0 or i == 1:
#         print(f"[{i:02d}/{N_ITER}] macro_f1={macro_f1:.5f} | best={best['macro_f1']:.5f}")

# elapsed = time.time() - t0
# print(f"Done. Elapsed: {elapsed:.1f}s")
# print("Best validation Macro F1:", best['macro_f1'])
# print("Best params:")
# print({k: v for k, v in best['params'].items() if k != 'n_jobs'})

# #  update best-found config
# best_lgb_params = best['params']

In [27]:
# best_lgb_params

In [28]:
# # Retrain your final model with the tuned params (requires train_lightgbm from the previous cell)
# model, train_preds, val_preds, test_preds = train_lightgbm(
#     train_pre_df, val_pre_df, test_pre_df, FEATURE_NAMES, best_lgb_params
# )

In [38]:
# import matplotlib.pyplot as plt

# print("--- Calibrating Threshold on Validation Set ---")

# thresholds = np.arange(0.1, 0.9, 0.01)
# f1_scores = []

# for thresh in thresholds:
#     # Use the validation set to find the best threshold
#     preds_bin = (val_preds > thresh).astype(int)
#     score = f1_score(val_pre_df['label'], preds_bin, average='macro')
#     f1_scores.append(score)

# best_idx = np.argmax(f1_scores)
# best_thresh = thresholds[best_idx]
# best_f1 = f1_scores[best_idx]

# print(f"Best Validation Macro F1: {best_f1:.4f} at Threshold: {best_thresh:.2f}")

# # Plot threshold curve
# plt.figure(figsize=(8, 4))
# plt.plot(thresholds, f1_scores, marker='o', markersize=3)
# plt.axvline(best_thresh, color='red', linestyle='--', label=f'Best Thresh: {best_thresh:.2f}')
# plt.title('Macro F1 vs. Decision Threshold')
# plt.xlabel('Threshold')
# plt.ylabel('Macro F1')
# plt.legend()
# plt.show()

In [61]:
# start_threshold = 0.95 #best 0.955 or 0.957

# for i in range(31):
#     threshold = 0.95 + i / 1000

#     submit_preds = (test_preds > threshold).astype(int)

#     submission = pd.DataFrame({
#         'ID': test_pre_df['ID'],
#         'label': submit_preds
#     })
#     filename = f"submission{i}.csv"

#     submission.to_csv(filename, index=False)

#     print("Distribution of predictions in submission:")
#     print(submission['label'].value_counts(normalize=True) * 100)
    
#     submit_solution(filename)

#     print("Wait 2 seconds...")
#     time.sleep(2)  
#     print("Done!")

In [60]:
# threshold = 0.96

# submit_preds = (test_preds > threshold).astype(int)

# submission = pd.DataFrame({
#     'ID': test_pre_df['ID'],
#     'label': submit_preds
# })

# submission.to_csv('submission.csv', index=False)

# print("Distribution of predictions in submission:")
# print(submission['label'].value_counts(normalize=True) * 100)